# Lab 3 EDA: Flight Delay Lakehouse

This notebook documents the lakehouse dataset and validates the outputs of the `bronze -> silver -> gold` pipeline.

Run the pipeline first:

```bash
python -m src.pipeline --skip-ml
```

or via Docker Compose:

```bash
docker compose up --build
```

In [1]:
from pathlib import Path

import polars as pl
from deltalake import DeltaTable

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
BRONZE_PATH = DATA_DIR / "bronze" / "flights"
SILVER_PATH = DATA_DIR / "silver" / "flights"
GOLD_ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"
GOLD_ANALYTICS_PATHS = {
    "by_airport": GOLD_ANALYTICS_DIR / "by_airport",
    "by_carrier": GOLD_ANALYTICS_DIR / "by_carrier",
    "by_hour": GOLD_ANALYTICS_DIR / "by_hour",
    "by_season": GOLD_ANALYTICS_DIR / "by_season",
}
GOLD_FEATURES_PATH = DATA_DIR / "gold" / "ml_features"

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

polars.config.Config

## Raw Dataset

The Kaggle source file is kept in `data/raw`. Because the downloaded dataset contains January 2024 only, the project simulates yearly arrival batches by splitting the source into `flights_2018.csv` ... `flights_2024.csv`.

In [2]:
raw_files = sorted(path.name for path in RAW_DIR.glob("*") if path.is_file())
raw_files

['flight_data.parquet',
 'flight_data_2018_2024.csv',
 'flights_2018.csv',
 'flights_2019.csv',
 'flights_2020.csv',
 'flights_2021.csv',
 'flights_2022.csv',
 'flights_2023.csv',
 'flights_2024.csv',
 'readme.html']

In [3]:
yearly_summary = []
for path in sorted(RAW_DIR.glob("flights_*.csv")):
    lf = pl.scan_csv(path)
    stats = (
        lf.select(
            pl.len().alias("rows"),
            pl.col("FL_DATE").min().alias("min_date"),
            pl.col("FL_DATE").max().alias("max_date"),
        )
        .collect()
        .to_dicts()[0]
    )
    stats["file"] = path.name
    yearly_summary.append(stats)

pl.DataFrame(yearly_summary).select(["file", "rows", "min_date", "max_date"])

file,rows,min_date,max_date
str,i64,str,str
"""flights_2018.csv""",83203,"""2018-01-01""","""2018-01-29"""
"""flights_2019.csv""",83203,"""2019-01-01""","""2019-01-31"""
"""flights_2020.csv""",83203,"""2020-01-01""","""2020-01-31"""
"""flights_2021.csv""",83203,"""2021-01-01""","""2021-01-31"""
"""flights_2022.csv""",83203,"""2022-01-01""","""2022-01-31"""
"""flights_2023.csv""",83203,"""2023-01-01""","""2023-01-31"""
"""flights_2024.csv""",83207,"""2024-01-01""","""2024-01-31"""


## Delta Table Versions

Delta transaction history confirms append writes, merges, overwrites, optimize operations, and time-travel-capable versions.

In [4]:
for name, path in [
    ("bronze", BRONZE_PATH),
    ("silver", SILVER_PATH),
    *[(f"gold_analytics_{name}", path) for name, path in GOLD_ANALYTICS_PATHS.items()],
    ("gold_features", GOLD_FEATURES_PATH),
]:
    dt = DeltaTable(str(path))
    print(name, {"version": dt.version(), "files": len(dt.files())})

bronze {'version': 13, 'files': 14}
silver {'version': 15, 'files': 7}
gold_analytics_by_airport {'version': 3, 'files': 1}
gold_analytics_by_carrier {'version': 3, 'files': 1}
gold_analytics_by_hour {'version': 3, 'files': 1}
gold_analytics_by_season {'version': 1, 'files': 1}
gold_features {'version': 3, 'files': 1}


In [5]:
silver_history = pl.DataFrame(DeltaTable(str(SILVER_PATH)).history())
silver_history.select(
    ["version", "timestamp", "operation", "operationParameters"]
).head(10)

version,timestamp,operation,operationParameters
i64,i64,str,struct[8]
15,1777213212272,"""OPTIMIZE""","{null,null,null,null,""104857600"",""[]"",null,null}"
14,1777213210435,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
13,1777213209077,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
12,1777213207728,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
11,1777213206354,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
10,1777213205073,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
9,1777213203754,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
8,1777213202237,"""MERGE""","{""[{""actionType"":""insert""}]"",""[]"",""target.flight_id = source.flight_id"",""[{""actionType"":""update""}]"",null,null,null,null}"
7,1777210721055,"""OPTIMIZE""","{null,null,null,null,""104857600"",""[]"",null,null}"


## Silver Table Quality Checks

Silver removes cancelled/diverted flights, drops rows without `ARR_DELAY`, normalizes categories, creates derived features, and partitions by `year` and `month`.

In [6]:
silver = pl.scan_delta(str(SILVER_PATH))

silver.select(
    pl.len().alias("rows"),
    pl.col("year").n_unique().alias("years"),
    pl.col("month").n_unique().alias("months"),
    pl.col("flight_id").n_unique().alias("unique_flight_ids"),
    pl.col("ARR_DELAY").null_count().alias("null_arr_delay"),
).collect()

rows,years,months,unique_flight_ids,null_arr_delay
u32,u32,u32,u32,u32
532450,7,1,532450,0


In [7]:
silver.group_by(["year", "month"]).agg(
    pl.len().alias("rows"),
    pl.col("ARR_DELAY").mean().alias("avg_arr_delay"),
    pl.col("is_delayed").mean().alias("delay_rate"),
).sort(["year", "month"]).collect()

year,month,rows,avg_arr_delay,delay_rate
i32,i8,u32,f64,f64
2018,1,78475,0.972475,0.188723
2019,1,76860,-2.422508,0.175878
2020,1,78152,-3.465618,0.160674
2021,1,75447,2.122417,0.214031
2022,1,72899,-1.356534,0.17826
2023,1,76314,1.219449,0.209267
2024,1,74303,4.431746,0.245172


In [8]:
silver.select(
    [
        "flight_date",
        "year",
        "month",
        "hour",
        "day_of_week",
        "season",
        "route",
        "ORIGIN",
        "DEST",
        "OP_UNIQUE_CARRIER",
        "ARR_DELAY",
        "is_delayed",
    ]
).head(10).collect()

flight_date,year,month,hour,day_of_week,season,route,ORIGIN,DEST,OP_UNIQUE_CARRIER,ARR_DELAY,is_delayed
datetime[μs],i32,i8,i64,i8,str,str,str,str,str,f64,i32
2021-01-24 00:00:00,2021,1,8,7,"""winter""","""HNL_OGG""","""HNL""","""OGG""","""HA""",-2.0,0
2021-01-09 00:00:00,2021,1,8,6,"""winter""","""HNL_OGG""","""HNL""","""OGG""","""HA""",4.0,0
2021-01-27 00:00:00,2021,1,8,3,"""winter""","""HNL_KOA""","""HNL""","""KOA""","""HA""",-13.0,0
2021-01-31 00:00:00,2021,1,8,7,"""winter""","""HNL_OGG""","""HNL""","""OGG""","""HA""",-7.0,0
2021-01-18 00:00:00,2021,1,8,1,"""winter""","""HNL_ITO""","""HNL""","""ITO""","""HA""",-10.0,0
2021-01-19 00:00:00,2021,1,8,2,"""winter""","""HNL_KOA""","""HNL""","""KOA""","""HA""",-8.0,0
2021-01-15 00:00:00,2021,1,8,5,"""winter""","""HNL_ITO""","""HNL""","""ITO""","""HA""",-4.0,0
2021-01-31 00:00:00,2021,1,8,7,"""winter""","""HNL_ITO""","""HNL""","""ITO""","""HA""",1.0,0
2021-01-15 00:00:00,2021,1,8,5,"""winter""","""HNL_LIH""","""HNL""","""LIH""","""HA""",-3.0,0


## Gold Analytics

Gold analytics are stored as separate Delta marts so each aggregate keeps its natural dimension column.

In [9]:
analytics_rows = []
for name, path in GOLD_ANALYTICS_PATHS.items():
    analytics_rows.append(
        {
            "mart": name,
            "rows": pl.scan_delta(str(path)).select(pl.len()).collect()[0, 0],
        }
    )
pl.DataFrame(analytics_rows)

mart,rows
str,i64
"""by_airport""",351
"""by_carrier""",21
"""by_hour""",24
"""by_season""",1


In [10]:
airport_analytics = pl.scan_delta(str(GOLD_ANALYTICS_PATHS["by_airport"]))
carrier_analytics = pl.scan_delta(str(GOLD_ANALYTICS_PATHS["by_carrier"]))

airport_analytics.sort("avg_arr_delay", descending=True).head(
    10
).collect(), carrier_analytics.sort("avg_arr_delay", descending=True).head(10).collect()

(shape: (10, 6)
 ┌────────┬───────────────┬──────────────────┬───────────────┬────────────┬─────────────┐
 │ ORIGIN ┆ avg_arr_delay ┆ median_arr_delay ┆ std_arr_delay ┆ delay_rate ┆ num_flights │
 │ ---    ┆ ---           ┆ ---              ┆ ---           ┆ ---        ┆ ---         │
 │ str    ┆ f64           ┆ f64              ┆ f64           ┆ f64        ┆ i32         │
 ╞════════╪═══════════════╪══════════════════╪═══════════════╪════════════╪═════════════╡
 │ ALO    ┆ 18.883721     ┆ 12.0             ┆ 36.245334     ┆ 0.465116   ┆ 43          │
 │ PUW    ┆ 15.467391     ┆ 8.5              ┆ 28.871618     ┆ 0.380435   ┆ 92          │
 │ AKN    ┆ 14.588235     ┆ -5.0             ┆ 41.18868      ┆ 0.352941   ┆ 17          │
 │ ART    ┆ 12.805556     ┆ -5.0             ┆ 36.96239      ┆ 0.333333   ┆ 36          │
 │ BET    ┆ 12.563636     ┆ 2.0              ┆ 28.208722     ┆ 0.363636   ┆ 55          │
 │ BRW    ┆ 12.521739     ┆ 4.0              ┆ 27.745384     ┆ 0.391304   ┆ 23      

## ML Feature Table

This table feeds both regression (`ARR_DELAY`) and classification (`is_delayed`) tasks.

In [11]:
features = pl.scan_delta(str(GOLD_FEATURES_PATH))
features.select(
    pl.len().alias("rows"),
    pl.col("ARR_DELAY").mean().alias("avg_arr_delay"),
    pl.col("ARR_DELAY").median().alias("median_arr_delay"),
    pl.col("is_delayed").mean().alias("delay_rate"),
).collect()

rows,avg_arr_delay,median_arr_delay,delay_rate
u32,f64,f64,f64
532450,0.193199,-6.0,0.195727


In [12]:
features.select(
    [
        "hour",
        "day_of_week",
        "month",
        "season",
        "DISTANCE",
        "DEP_DELAY",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "ARR_DELAY",
        "is_delayed",
    ]
).head(10).collect()

hour,day_of_week,month,season,DISTANCE,DEP_DELAY,OP_UNIQUE_CARRIER,ORIGIN,DEST,ARR_DELAY,is_delayed
i64,i8,i8,str,f64,f64,str,str,str,f64,i32
8,2,1,"""winter""",100.0,2.0,"""HA""","""HNL""","""OGG""",13.0,0
8,4,1,"""winter""",216.0,-4.0,"""HA""","""HNL""","""ITO""",-8.0,0
8,1,1,"""winter""",100.0,22.0,"""HA""","""HNL""","""OGG""",24.0,1
8,7,1,"""winter""",163.0,-4.0,"""HA""","""HNL""","""KOA""",-7.0,0
8,1,1,"""winter""",102.0,-5.0,"""HA""","""HNL""","""LIH""",-7.0,0
8,3,1,"""winter""",163.0,-3.0,"""HA""","""HNL""","""KOA""",-9.0,0
8,3,1,"""winter""",100.0,4.0,"""HA""","""HNL""","""OGG""",0.0,0
8,7,1,"""winter""",102.0,-4.0,"""HA""","""HNL""","""LIH""",-2.0,0
8,3,1,"""winter""",163.0,-4.0,"""HA""","""HNL""","""KOA""",-8.0,0


## Polars Lazy Explain

This query demonstrates projection and selection pushdown. The same output is stored in `logs/polars_explain.txt`.

In [13]:
query = (
    pl.scan_delta(str(SILVER_PATH))
    .filter((pl.col("year") == 2024) & (pl.col("ARR_DELAY").is_not_null()))
    .select(["year", "month", "ORIGIN", "ARR_DELAY", "is_delayed"])
    .group_by(["year", "month", "ORIGIN"])
    .agg(
        [
            pl.col("ARR_DELAY").mean().alias("avg_arr_delay"),
            pl.col("is_delayed").mean().alias("delay_rate"),
            pl.len().alias("num_flights"),
        ]
    )
    .sort("avg_arr_delay", descending=True)
)

print(query.explain())

SORT BY [col("avg_arr_delay")]
  AGGREGATE
  	[col("ARR_DELAY").mean().alias("avg_arr_delay"), col("is_delayed").mean().alias("delay_rate"), len().alias("num_flights")] BY [col("year"), col("month"), col("ORIGIN")] FROM

      PYTHON SCAN 
      PROJECT 5/21 COLUMNS
      SELECTION: [([(col("year")) == (2024)]) & (col("ARR_DELAY").is_not_null())]


Expected evidence in the explain output:

```text
PROJECT 5/21 COLUMNS
SELECTION: [([(col("year")) == (2024)]) & (col("ARR_DELAY").is_not_null())]
```